'''
Tipagem Correta: Converter DATA para o tipo Date, e as métricas numéricas (KM_INICIAL, KM_FINAL, PAINEL_L_100KM, VELOC_MEDIA_KM_H) de String para Float/Double (já que vieram como texto da Bronze).

Criação de Colunas Derivadas (Enriquecimento): Calcular automaticamente a Quilometragem Rodada (KM_FINAL - KM_INICIAL) em cada viagem.

Tratamento de Nulos: Tratar dias sem registro ou valores vazios de forma limpa.

faz a leitura da camada Bronze, tipa os dados corretamente, calcula a quilometragem rodada e salva a tabela Silver de forma limpa
'''

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

catalogo = "will"
esquema = "i30"

tabela_bronze = f"{catalogo}.{esquema}.bronze_viagens"
tabela_silver = f"{catalogo}.{esquema}.silver_viagens"

# 1. Leitura dos dados da Bronze
df_bronze = spark.table(tabela_bronze)

# 2. Transformações e padronização para a Camada Silver
df_silver = (
    df_bronze
    .withColumn("DATA", F.to_date(F.col("DATA"), "dd/MM/yyyy"))
    # Mapeia os nomes reais da Bronze para os nomes padronizados da Silver
    .withColumn("KM_INICIAL", F.col("INICIAL").cast(IntegerType()))
    .withColumn("KM_FINAL", F.col("FINAL").cast(IntegerType()))
    .withColumn("PAINEL_L_100KM", F.col("PAINEL_L_100KM").cast(DoubleType()))
    .withColumn("VELOC_MEDIA_KM_H", F.col("KM_H").cast(IntegerType()))
    # Enriquecimento: Cálculo dos KM rodados
    .withColumn(
        "KM_RODADAS", 
        F.when((F.col("FINAL").isNotNull()) & (F.col("INICIAL").isNotNull()), 
               F.col("FINAL").cast(IntegerType()) - F.col("INICIAL").cast(IntegerType()))
         .otherwise(None)
    )
    # Seleciona apenas as colunas limpas e oficiais da Silver
    .select(
        "DATA", 
        "KM_INICIAL", 
        "KM_FINAL", 
        "KM_RODADAS", 
        "PAINEL_L_100KM", 
        "VELOC_MEDIA_KM_H",
        "_source_file"
    )
    .withColumn("_processed_at", F.current_timestamp())
)

# 3. Gravação na Camada Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)

print(f"Camada Silver gravada com sucesso: {tabela_silver}")
display(spark.table(tabela_silver))